In [1]:
# # Dataset Generator for Composite DNA - Cross-Platform Robustness Study
# ## New Sequencing Platforms: Nanopore (R21, B22, NP22, NPF22) + Newer Illumina (BOS22)
# ## Each profile uses its standard sequence length from the corresponding original dataset

# In[1]:

# =============================================================================
# CELL 1: IMPORTS
# =============================================================================
import random
import numpy as np
import pickle
import json
import os
from collections import Counter
from datetime import datetime
import time

In [2]:
# In[2]:

# =============================================================================
# CELL 2: CONFIGURATION
# =============================================================================

# ------------------- SELECT ERROR MODEL -------------------
# NEW PLATFORM OPTIONS:
#   "R21"    -> Oxford Nanopore MinION + Twist Bioscience (Rang et al. 2021)
#   "B22"    -> Nanopore MinION Short-read + Twist Bioscience (Bar-Lev et al. 2022)
#   "BOS22"  -> Illumina MiSeq 2022 + Twist Bioscience (very low error, newer Illumina)
#   "NP22"   -> Nanopore Pilot Nov-2022 + Twist Bioscience (highly non-uniform across bases)
#   "NPF22"  -> Nanopore Full Pool Nov-2022 + Twist Bioscience (comprehensive Nanopore)
#
# (Original profiles for reference, already have results: "erlich"/"EZ17", "grass"/"G15", "organick"/"O17")
# ----------------------------------------------------------
ERROR_MODEL = "R21"  # <-- CHANGE THIS TO SELECT ERROR MODEL

# ------------------- SELECT ALPHABET MODE -------------------
# Options: "2mix_only", "2mix_3mix", "2mix_3mix_4mix"
ALPHABET_MODE = "2mix_3mix"  # <-- CHANGE THIS TO SELECT ALPHABET MODE
# ------------------------------------------------------------

# All new profiles derive from the same Erlich/Twist oligo pool (72,000 oligos of 152nt,
# synthesized by Twist Bioscience), sequenced with different technologies.
# Standard oligo design: 152nt total, 16nt index/primer region → 136nt payload.
# This matches EZ17's design since the same physical oligo pool was reused.
#
# Oligo Design Summary (standard lengths per original dataset):
#   Original profiles (different oligo pools / designs):
#     EZ17 : 152nt Twist, 16nt index  → seq_length = 136  (Erlich & Zielinski 2017)
#     G15  : 117nt CustomArray, 13nt index → seq_length = 104  (Grass et al. 2015)
#     O17  : 110nt Twist, 33nt index  → seq_length = 77   (Organick et al. 2018)
#
#   New profiles (same Erlich/Twist 152nt pool, different sequencing technologies):
#     R21  : 152nt Twist, 16nt index  → seq_length = 136  (Rang et al. 2021, Nanopore MinION)
#     B22  : 152nt Twist, 16nt index  → seq_length = 136  (Bar-Lev et al. 2022, MinION Short)
#     BOS22: 152nt Twist, 16nt index  → seq_length = 136  (Bar-Lev & Sabary 2022, Illumina MiSeq 2022)
#     NP22 : 152nt Twist, 16nt index  → seq_length = 136  (Bar-Lev & Sabary 2022, Nanopore Pilot)
#     NPF22: 152nt Twist, 16nt index  → seq_length = 136  (Bar-Lev & Sabary 2022, Nanopore Full)

ERROR_MODEL_SPECS = {
    # ---------- NEW NANOPORE PROFILES ----------
    "R21": {
        "full_length": 152,
        "index_length": 16,
        "seq_length": 136,  # 152 - 16 (same Erlich/Twist pool)
        "name": "R21",
        "platform": "Oxford Nanopore MinION",
        "synthesis": "Twist Bioscience",
        "reference": "Rang et al. 2021",
        "notes": "Standard MinION, high insertion rate (~1.65%)"
    },
    "B22": {
        "full_length": 152,
        "index_length": 16,
        "seq_length": 136,  # 152 - 16 (same Erlich/Twist pool)
        "name": "B22",
        "platform": "Nanopore MinION Short",
        "synthesis": "Twist Bioscience",
        "reference": "Bar-Lev et al. 2022",
        "notes": "Short-read MinION, balanced errors (~1.1% each)"
    },
    "NP22": {
        "full_length": 152,
        "index_length": 16,
        "seq_length": 136,  # 152 - 16 (same Erlich/Twist pool)
        "name": "NP22",
        "platform": "Nanopore Pilot Nov-2022",
        "synthesis": "Twist Bioscience",
        "reference": "Bar-Lev & Sabary et al. 2022",
        "notes": "Highly non-uniform per-base errors, C and T dominated"
    },
    "NPF22": {
        "full_length": 152,
        "index_length": 16,
        "seq_length": 136,  # 152 - 16 (same Erlich/Twist pool)
        "name": "NPF22",
        "platform": "Nanopore Full Pool Nov-2022",
        "synthesis": "Twist Bioscience",
        "reference": "Bar-Lev & Sabary et al. 2022 (updated)",
        "notes": "Full pool Nanopore, high uniform errors (~1.5%)"
    },
    # ---------- NEW ILLUMINA PROFILE ----------
    "BOS22": {
        "full_length": 152,
        "index_length": 16,
        "seq_length": 136,  # 152 - 16 (same Erlich/Twist pool)
        "name": "BOS22",
        "platform": "Illumina MiSeq 2022",
        "synthesis": "Twist Bioscience",
        "reference": "Bar-Lev & Sabary et al. 2022",
        "notes": "Ultra-low error rates (~0.05%), newer Illumina generation"
    },
}

CONFIG = {
    # Error Model
    "error_model": ERROR_MODEL,
    "error_specs": ERROR_MODEL_SPECS[ERROR_MODEL],
    
    # Alphabet Mode
    "alphabet_mode": ALPHABET_MODE,
    
    # Dataset Parameters
    "num_samples": 100000,
    "seq_length": ERROR_MODEL_SPECS[ERROR_MODEL]["seq_length"],
    "max_coverage": 25,
    
    # Output Directory
    "dataset_dir": "./dataset_cross_platform",
    
    # Reproducibility
    "seed": 42
}

# Set vocab_size based on alphabet mode
VOCAB_SIZES = {
    "2mix_only": 10,
    "2mix_3mix": 14,
    "2mix_3mix_4mix": 15
}

CONFIG["vocab_size"] = VOCAB_SIZES[CONFIG["alphabet_mode"]]

# Create dataset name including error model
dataset_name = f"dna_{CONFIG['error_specs']['name']}_{CONFIG['alphabet_mode']}"
CONFIG["dataset_path"] = (f"{CONFIG['dataset_dir']}/"
                          f"{dataset_name}_"
                          f"{CONFIG['num_samples']}_{CONFIG['max_coverage']}.pkl")

os.makedirs(CONFIG['dataset_dir'], exist_ok=True)

print(f"{'='*70}")
print(f"📋 CROSS-PLATFORM DATASET GENERATION CONFIGURATION")
print(f"{'='*70}")
print(f"   Error Model: {CONFIG['error_model']} ({CONFIG['error_specs']['name']})")
print(f"   Platform: {CONFIG['error_specs']['platform']}")
print(f"   Synthesis: {CONFIG['error_specs']['synthesis']}")
print(f"   Reference: {CONFIG['error_specs']['reference']}")
print(f"   Notes: {CONFIG['error_specs']['notes']}")
print(f"   Oligo Design: {CONFIG['error_specs']['full_length']}nt total, "
      f"{CONFIG['error_specs']['index_length']}nt index → "
      f"{CONFIG['error_specs']['seq_length']}nt payload (standard)")
print(f"   Sequence Length: {CONFIG['seq_length']}")
print(f"   Alphabet Mode: {CONFIG['alphabet_mode']}")
print(f"   Vocab Size: {CONFIG['vocab_size']} classes")
print(f"   Num Samples: {CONFIG['num_samples']:,}")
print(f"   Max Coverage: {CONFIG['max_coverage']}")
print(f"   Output Path: {CONFIG['dataset_path']}")
print(f"{'='*70}")

📋 CROSS-PLATFORM DATASET GENERATION CONFIGURATION
   Error Model: R21 (R21)
   Platform: Oxford Nanopore MinION
   Synthesis: Twist Bioscience
   Reference: Rang et al. 2021
   Notes: Standard MinION, high insertion rate (~1.65%)
   Oligo Design: 152nt total, 16nt index → 136nt payload (standard)
   Sequence Length: 136
   Alphabet Mode: 2mix_3mix
   Vocab Size: 14 classes
   Num Samples: 100,000
   Max Coverage: 25
   Output Path: ./dataset_cross_platform/dna_R21_2mix_3mix_100000_25.pkl


In [3]:
# In[3]:

# =============================================================================
# CELL 3: SEED FOR REPRODUCIBILITY
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: dataset_generator_2mix_3mix-Erlich.py → Cell 3
# Copy: set_seed() function
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

set_seed(CONFIG['seed'])
print(f"🎲 Random seed set to: {CONFIG['seed']}")



🎲 Random seed set to: 42


In [4]:
# In[4]:

# =============================================================================
# CELL 4: COMPOSITE DNA ALPHABET DEFINITIONS
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: dataset_generator_2mix_3mix-Erlich.py → Cell 4
# Copy: PURE_BASES, TWO_MIX_MAP, THREE_MIX_MAP, FOUR_MIX_MAP dicts
# Copy: build_composite_map(), build_symbol_to_idx(), build_ideal_vectors()
# Copy: The print block at the end of Cell 4
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

# ----- Pure Bases -----
PURE_BASES = {
    'A': ['A'],
    'C': ['C'],
    'G': ['G'],
    'T': ['T'],
}

# ----- Two-Nucleotide Mixtures (6 total) -----
TWO_MIX_MAP = {
    'M1': ['A', 'T'],
    'M2': ['C', 'G'],
    'M3': ['C', 'T'],
    'M4': ['G', 'T'],
    'M5': ['A', 'C'],
    'M6': ['A', 'G'],
}

# ----- Three-Nucleotide Mixtures (4 total) -----
THREE_MIX_MAP = {
    'T1': ['A', 'C', 'G'],
    'T2': ['A', 'C', 'T'],
    'T3': ['A', 'G', 'T'],
    'T4': ['C', 'G', 'T'],
}

# ----- Four-Nucleotide Mixture (1 total) -----
FOUR_MIX_MAP = {
    'Q1': ['A', 'C', 'G', 'T'],
}


def build_composite_map(mode):
    """Build the composite symbol mapping based on alphabet mode."""
    composite_map = PURE_BASES.copy()
    composite_map.update(TWO_MIX_MAP)
    if mode in ["2mix_3mix", "2mix_3mix_4mix"]:
        composite_map.update(THREE_MIX_MAP)
    if mode == "2mix_3mix_4mix":
        composite_map.update(FOUR_MIX_MAP)
    return composite_map


def build_symbol_to_idx(mode):
    """Build symbol-to-index mapping based on alphabet mode."""
    symbol_to_idx = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    symbol_to_idx.update({'M1': 4, 'M2': 5, 'M3': 6, 'M4': 7, 'M5': 8, 'M6': 9})
    if mode in ["2mix_3mix", "2mix_3mix_4mix"]:
        symbol_to_idx.update({'T1': 10, 'T2': 11, 'T3': 12, 'T4': 13})
    if mode == "2mix_3mix_4mix":
        symbol_to_idx.update({'Q1': 14})
    return symbol_to_idx


def build_ideal_vectors(mode):
    """Build ideal frequency vectors for all symbols."""
    ideal_vectors = [
        [1.0, 0.0, 0.0, 0.0],  # A
        [0.0, 1.0, 0.0, 0.0],  # C
        [0.0, 0.0, 1.0, 0.0],  # G
        [0.0, 0.0, 0.0, 1.0],  # T
        [0.5, 0.0, 0.0, 0.5],  # M1 (A|T)
        [0.0, 0.5, 0.5, 0.0],  # M2 (C|G)
        [0.0, 0.5, 0.0, 0.5],  # M3 (C|T)
        [0.0, 0.0, 0.5, 0.5],  # M4 (G|T)
        [0.5, 0.5, 0.0, 0.0],  # M5 (A|C)
        [0.5, 0.0, 0.5, 0.0],  # M6 (A|G)
    ]
    if mode in ["2mix_3mix", "2mix_3mix_4mix"]:
        third = 1.0 / 3.0
        ideal_vectors.extend([
            [third, third, third, 0.0],    # T1
            [third, third, 0.0, third],    # T2
            [third, 0.0, third, third],    # T3
            [0.0, third, third, third],    # T4
        ])
    if mode == "2mix_3mix_4mix":
        ideal_vectors.append([0.25, 0.25, 0.25, 0.25])  # Q1
    return ideal_vectors


# Build mappings for current mode
COMPOSITE_MAP = build_composite_map(CONFIG["alphabet_mode"])
SYMBOL_TO_IDX = build_symbol_to_idx(CONFIG["alphabet_mode"])
IDEAL_VECTORS = build_ideal_vectors(CONFIG["alphabet_mode"])
ALL_SYMBOLS = list(COMPOSITE_MAP.keys())

print(f"\n🧬 Composite Alphabet ({CONFIG['alphabet_mode']}):")
print(f"   Total Symbols: {len(ALL_SYMBOLS)}")
print(f"\n   {'Symbol':<8} {'Index':<6} {'Composition':<15} {'Ideal Vector [A,C,G,T]'}")
print(f"   {'-'*60}")
for sym in ALL_SYMBOLS:
    idx = SYMBOL_TO_IDX[sym]
    bases = ' | '.join(COMPOSITE_MAP[sym])
    vec = IDEAL_VECTORS[idx]
    vec_str = f"[{vec[0]:.3f}, {vec[1]:.3f}, {vec[2]:.3f}, {vec[3]:.3f}]"
    print(f"   {sym:<8} {idx:<6} {bases:<15} {vec_str}")






🧬 Composite Alphabet (2mix_3mix):
   Total Symbols: 14

   Symbol   Index  Composition     Ideal Vector [A,C,G,T]
   ------------------------------------------------------------
   A        0      A               [1.000, 0.000, 0.000, 0.000]
   C        1      C               [0.000, 1.000, 0.000, 0.000]
   G        2      G               [0.000, 0.000, 1.000, 0.000]
   T        3      T               [0.000, 0.000, 0.000, 1.000]
   M1       4      A | T           [0.500, 0.000, 0.000, 0.500]
   M2       5      C | G           [0.000, 0.500, 0.500, 0.000]
   M3       6      C | T           [0.000, 0.500, 0.000, 0.500]
   M4       7      G | T           [0.000, 0.000, 0.500, 0.500]
   M5       8      A | C           [0.500, 0.500, 0.000, 0.000]
   M6       9      A | G           [0.500, 0.000, 0.500, 0.000]
   T1       10     A | C | G       [0.333, 0.333, 0.333, 0.000]
   T2       11     A | C | T       [0.333, 0.333, 0.000, 0.333]
   T3       12     A | G | T       [0.333, 0.000, 0.3

In [5]:
# In[5]:

# =============================================================================
# CELL 5: ERROR RATES CLASS - CROSS-PLATFORM VERSION
# =============================================================================
# This extends the original ErrorRates class with 5 new profiles from
# the other_github_dataset repository, covering Nanopore and newer Illumina.

class ErrorRates:
    """Error rate configuration for multiple DNA sequencing technologies.
    
    All profiles use their standard oligo-derived sequence lengths.
    
    Original profiles (already evaluated, different oligo designs):
        EZ17  - Illumina MiSeq + Twist         (Erlich & Zielinski 2017)    152nt → n=136
        G15   - Illumina MiSeq + CustomArray    (Grass et al. 2015)         117nt → n=104
        O17   - Illumina NextSeq + Twist        (Organick et al. 2018)      110nt → n=77
    
    NEW cross-platform profiles (same 152nt Erlich/Twist pool, different sequencing):
        R21   - Oxford Nanopore MinION + Twist         (Rang et al. 2021)           152nt → n=136
        B22   - Nanopore MinION Short + Twist          (Bar-Lev et al. 2022)        152nt → n=136
        BOS22 - Illumina MiSeq 2022 + Twist            (Bar-Lev & Sabary 2022)     152nt → n=136
        NP22  - Nanopore Pilot Nov-2022 + Twist        (Bar-Lev & Sabary 2022)      152nt → n=136
        NPF22 - Nanopore Full Pool Nov-2022 + Twist    (Bar-Lev & Sabary 2022)      152nt → n=136
    """
    
    def __init__(self):
        self.general_errors = {'d': 0.0, 'ld': 0.0, 'i': 0.0, 's': 0.0}
        self.per_base_errors = {
            'A': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0},
            'C': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0},
            'G': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0},
            'T': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0}
        }
    
    # =====================================================================
    # ORIGINAL PROFILES (kept for reference / combined runs)
    # =====================================================================
    
    def set_EZ17_values(self):
        """Erlich & Zielinski 2017 (Illumina MiSeq + Twist) error profile."""
        print("   >> Loading Erlich (EZ17) Error Profile...")
        self.general_errors = {'s': 1.32e-03, 'i': 5.81e-04, 'd': 9.58e-04, 'ld': 2.33e-04}
        self.per_base_errors['A'] = {'s': 0.00135, 'i': 0.00057, 'pi': 0.00059, 'd': 0.00099, 'ld': 0.00024}
        self.per_base_errors['C'] = {'s': 0.00135, 'i': 0.00059, 'pi': 0.00058, 'd': 0.00098, 'ld': 0.00023}
        self.per_base_errors['G'] = {'s': 0.00126, 'i': 0.00059, 'pi': 0.00057, 'd': 0.00094, 'ld': 0.00023}
        self.per_base_errors['T'] = {'s': 0.00132, 'i': 0.00058, 'pi': 0.00058, 'd': 0.00096, 'ld': 0.00023}
    
    def set_G15_values(self):
        """Grass et al. 2015 (Illumina MiSeq + CustomArray) error profile."""
        print("   >> Loading Grass (G15) Error Profile...")
        self.general_errors = {'s': 5.84e-03, 'i': 8.57e-04, 'd': 5.37e-03, 'ld': 3.48e-04}
        self.per_base_errors['A'] = {'s': 0.00605, 'i': 0.0009,  'pi': 0.00092, 'd': 0.00543, 'ld': 0.00036}
        self.per_base_errors['C'] = {'s': 0.00563, 'i': 0.00083, 'pi': 0.00081, 'd': 0.00513, 'ld': 0.00034}
        self.per_base_errors['G'] = {'s': 0.00577, 'i': 0.00085, 'pi': 0.00087, 'd': 0.00539, 'ld': 0.00034}
        self.per_base_errors['T'] = {'s': 0.00591, 'i': 0.00084, 'pi': 0.00084, 'd': 0.00559, 'ld': 0.00036}
    
    def set_O17_values(self):
        """Organick et al. 2018 (Illumina NextSeq + Twist) error profile."""
        print("   >> Loading Organick (O17) Error Profile...")
        self.general_errors = {'s': 2.52e-03, 'i': 4.14e-04, 'd': 6.94e-04, 'ld': 2.11e-04}
        self.per_base_errors['A'] = {'s': 0.00717, 'i': 0.0003,  'pi': 0.0012,  'd': 0.00201, 'ld': 0.00054}
        self.per_base_errors['C'] = {'s': 0.00034, 'i': 0.00007, 'pi': 0.00007, 'd': 0.00006, 'ld': 0.00001}
        self.per_base_errors['G'] = {'s': 0.00196, 'i': 0.00125, 'pi': 0.00029, 'd': 0.00058, 'ld': 0.00023}
        self.per_base_errors['T'] = {'s': 0.00055, 'i': 0.00006, 'pi': 0.00009, 'd': 0.00014, 'ld': 0.00006}
    
    # =====================================================================
    # NEW NANOPORE PROFILES
    # =====================================================================
    
    def set_R21_values(self):
        """Rang et al. 2021: Oxford Nanopore MinION + Twist Bioscience.
        
        Characteristics:
        - Insertion-dominated: ~1.65% mean insertion rate (highest among all profiles)
        - Substitution: ~1.08%, Deletion: ~1.18%
        - Relatively uniform across bases (unlike O17)
        - ~10x higher total error rate than EZ17
        """
        print("   >> Loading R21 (Nanopore MinION + Twist) Error Profile...")
        self.general_errors = {
            's': 1.08e-02,
            'i': 1.65e-02,
            'd': 1.18e-02,
            'ld': 3.36e-03
        }
        self.per_base_errors['A'] = {'s': 1.039e-02, 'i': 1.61e-02,  'pi': 1.585e-02, 'd': 1.192e-02, 'ld': 0.314e-02}
        self.per_base_errors['C'] = {'s': 1.042e-02, 'i': 1.639e-02, 'pi': 1.578e-02, 'd': 1.26e-02,  'ld': 0.334e-02}
        self.per_base_errors['G'] = {'s': 1.094e-02, 'i': 1.604e-02, 'pi': 1.7e-02,   'd': 1.267e-02, 'ld': 0.337e-02}
        self.per_base_errors['T'] = {'s': 1.131e-02, 'i': 1.738e-02, 'pi': 1.729e-02, 'd': 1.327e-02, 'ld': 0.357e-02}
    
    def set_B22_values(self):
        """Bar-Lev et al. 2022: Nanopore MinION Short-read + Twist Bioscience.
        
        Characteristics:
        - Balanced error profile: sub ~1.12%, ins ~1.08%, del ~0.79%
        - Most uniform per-base distribution among Nanopore profiles
        - Lower total error than R21 (short-read protocol advantage)
        """
        print("   >> Loading B22 (Nanopore MinION Short + Twist) Error Profile...")
        self.general_errors = {
            's': 1.12e-02,
            'i': 1.08e-02,
            'd': 7.87e-03,
            'ld': 2.05e-03
        }
        self.per_base_errors['A'] = {'s': 0.01146, 'i': 0.01104, 'pi': 0.01096, 'd': 0.00798, 'ld': 0.0021}
        self.per_base_errors['C'] = {'s': 0.01117, 'i': 0.01092, 'pi': 0.01088, 'd': 0.00804, 'ld': 0.00208}
        self.per_base_errors['G'] = {'s': 0.01129, 'i': 0.01073, 'pi': 0.01071, 'd': 0.00792, 'ld': 0.00201}
        self.per_base_errors['T'] = {'s': 0.01094, 'i': 0.01039, 'pi': 0.01053, 'd': 0.00769, 'ld': 0.00201}
    
    def set_NP22_values(self):
        """Nanopore Pilot Pool Nov-2022 + Twist Bioscience (BOS22PILOTOMER).
        
        Characteristics:
        - HIGHLY non-uniform per-base errors (most extreme variation)
        - T has ~5x higher substitution than A (2.44% vs 0.52%)
        - C has ~2x higher deletion than A (1.18% vs 0.37%)
        - Tests decoder robustness to asymmetric error landscapes
        """
        print("   >> Loading NP22 (Nanopore Pilot Nov-2022 + Twist) Error Profile...")
        self.general_errors = {
            's': 1.29e-02,
            'i': 1.16e-02,
            'd': 9.75e-03,
            'ld': 2.82e-03
        }
        self.per_base_errors['A'] = {'s': 0.52e-2,  'i': 0.644e-2, 'pi': 0.582e-2, 'd': 0.373e-2, 'ld': 0.098e-2}
        self.per_base_errors['C'] = {'s': 1.498e-2, 'i': 1.331e-2, 'pi': 1.34e-2,  'd': 1.178e-2, 'ld': 0.336e-2}
        self.per_base_errors['G'] = {'s': 0.638e-2, 'i': 0.735e-2, 'pi': 0.667e-2, 'd': 0.474e-2, 'ld': 0.128e-2}
        self.per_base_errors['T'] = {'s': 2.437e-2, 'i': 1.867e-2, 'pi': 1.983e-2, 'd': 1.903e-2, 'ld': 0.55e-2}
    
    def set_NPF22_values(self):
        """Nanopore Full Pool Nov-2022 (updated) + Twist Bioscience.
        
        Characteristics:
        - Comprehensive full-pool Nanopore data
        - Highest overall error rates: sub ~1.56%, ins ~1.24%, del ~0.98%
        - More uniform than NP22 pilot, but higher magnitude
        - Stress-tests decoder at extreme noise levels
        """
        print("   >> Loading NPF22 (Nanopore Full Pool Nov-2022 + Twist) Error Profile...")
        self.general_errors = {
            's': 1.56e-02,
            'i': 1.24e-02,
            'd': 9.79e-03,
            'ld': 2.67e-03
        }
        self.per_base_errors['A'] = {'s': 1.7e-2,   'i': 1.32e-2,  'pi': 1.343e-2, 'd': 1.095e-2, 'ld': 0.294e-2}
        self.per_base_errors['C'] = {'s': 1.606e-2, 'i': 1.271e-2, 'pi': 1.272e-2, 'd': 1.038e-2, 'ld': 0.277e-2}
        self.per_base_errors['G'] = {'s': 1.593e-2, 'i': 1.256e-2, 'pi': 1.255e-2, 'd': 1.014e-2, 'ld': 0.273e-2}
        self.per_base_errors['T'] = {'s': 1.358e-2, 'i': 1.116e-2, 'pi': 1.094e-2, 'd': 0.849e-2, 'ld': 0.224e-2}
    
    # =====================================================================
    # NEW ILLUMINA PROFILE
    # =====================================================================
    
    def set_BOS22_values(self):
        """Illumina MiSeq 2022 + Twist Bioscience (second pilot, 08-09-2022).
        
        Characteristics:
        - Ultra-low error rates (~10x lower than EZ17)
        - sub ~0.053%, ins ~0.005%, del ~0.007%
        - Non-uniform: T dominates substitution (0.175% vs A's 0.004%)
        - Tests decoder when errors are near-zero (ceiling effect expected)
        """
        print("   >> Loading BOS22 (Illumina MiSeq 2022 + Twist) Error Profile...")
        self.general_errors = {
            's': 5.29e-04,
            'i': 5.42e-05,
            'd': 7.08e-05,
            'ld': 1.81e-05
        }
        # Note: some per-base rates are effectively 0 for A and G
        self.per_base_errors['A'] = {'s': 0.004e-2,  'i': 0.004e-2, 'pi': 0.0,     'd': 0.0,     'ld': 0.0}
        self.per_base_errors['C'] = {'s': 0.024e-2,  'i': 0.006e-2, 'pi': 0.002e-2, 'd': 0.003e-2, 'ld': 0.001e-2}
        self.per_base_errors['G'] = {'s': 0.004e-2,  'i': 0.005e-2, 'pi': 0.0,     'd': 0.0,     'ld': 0.0}
        self.per_base_errors['T'] = {'s': 0.175e-2,  'i': 0.007e-2, 'pi': 0.018e-2, 'd': 0.024e-2, 'ld': 0.006e-2}
    
    # =====================================================================
    # DISPATCHER
    # =====================================================================
    
    def set_values_by_model(self, model_name):
        """Set error values based on model name string."""
        dispatch = {
            # Original profiles
            "erlich": self.set_EZ17_values,
            "EZ17":   self.set_EZ17_values,
            "grass":  self.set_G15_values,
            "G15":    self.set_G15_values,
            "organick": self.set_O17_values,
            "O17":    self.set_O17_values,
            # New profiles
            "R21":    self.set_R21_values,
            "B22":    self.set_B22_values,
            "BOS22":  self.set_BOS22_values,
            "NP22":   self.set_NP22_values,
            "NPF22":  self.set_NPF22_values,
        }
        if model_name not in dispatch:
            raise ValueError(
                f"Unknown error model: '{model_name}'. "
                f"Available: {list(dispatch.keys())}"
            )
        dispatch[model_name]()
    
    def print_current_values(self):
        """Print current error configuration with summary statistics."""
        print("\n   --- Error Configuration ---")
        print(f"   General: sub={self.general_errors['s']:.5f}, "
              f"ins={self.general_errors['i']:.5f}, "
              f"del={self.general_errors['d']:.5f}, "
              f"ldel={self.general_errors['ld']:.5f}")
        total_general = sum(self.general_errors.values())
        print(f"   Total error rate: {total_general:.5f} ({total_general*100:.3f}%)")
        print(f"   {'Base':<6} {'Sub':>10} {'Ins':>10} {'Del':>10} {'LDel':>10}")
        print(f"   {'-'*46}")
        for base in ['A', 'C', 'G', 'T']:
            r = self.per_base_errors[base]
            print(f"   {base:<6} {r['s']:>10.5f} {r['i']:>10.5f} {r['d']:>10.5f} {r['ld']:>10.5f}")
        print(f"   {'-'*46}")
    
    def get_error_summary(self):
        """Return a dict summary for metadata storage."""
        return {
            'general': dict(self.general_errors),
            'per_base': {b: dict(v) for b, v in self.per_base_errors.items()},
            'total_rate': sum(self.general_errors.values())
        }

In [6]:
# In[6]:

# =============================================================================
# CELL 6: SEQUENCE GENERATION FUNCTIONS
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: dataset_generator_2mix_3mix-Erlich.py → Cell 6
# Copy: generate_composite_sequence(), realize_sequence(), apply_ids_noise()
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

def generate_composite_sequence(length, composite_map):
    """Generates a random sequence of composite symbols."""
    symbols = list(composite_map.keys())
    return [random.choice(symbols) for _ in range(length)]


def realize_sequence(composite_seq, composite_map):
    """Converts composite symbols to a single DNA realization."""
    realized = []
    for sym in composite_seq:
        nucleotide = random.choice(composite_map[sym])
        realized.append(nucleotide)
    return "".join(realized)


def apply_ids_noise(sequence, error_profile):
    """Apply Insertion, Deletion, Substitution noise to a DNA sequence."""
    bases = ['A', 'C', 'G', 'T']
    noisy_seq = []
    
    for base in sequence:
        if base not in bases:
            continue
        
        rates = error_profile.per_base_errors[base]
        p_sub = rates['s']
        p_ins = rates['i']
        p_del = rates['d']
        
        # 1. DELETION Check
        if random.random() < p_del:
            continue
            
        # 2. INSERTION Check (pre-insertion)
        if random.random() < p_ins:
            noisy_seq.append(random.choice(bases))
            
        # 3. SUBSTITUTION vs MATCH Check
        if random.random() < p_sub:
            options = [b for b in bases if b != base]
            noisy_seq.append(random.choice(options))
        else:
            noisy_seq.append(base)
            
    return "".join(noisy_seq)


In [7]:
# In[7]:

# =============================================================================
# CELL 7: MAIN DATASET GENERATION FUNCTION
# =============================================================================
# This is MODIFIED from the original to use dynamic error model selection.

def generate_dataset(config, composite_map, symbol_to_idx, ideal_vectors):
    """
    Generate dataset with extended composite alphabet for any error model.
    
    Key difference from original: Uses config['error_model'] to dynamically
    select the error profile instead of hardcoding EZ17.
    """
    # Setup error profile - DYNAMIC SELECTION
    errors = ErrorRates()
    errors.set_values_by_model(config['error_model'])
    errors.print_current_values()
    
    num_samples = config['num_samples']
    seq_length = config['seq_length']
    coverage = config['max_coverage']
    filename = config['dataset_path']
    
    error_specs = config['error_specs']
    
    # Initialize dataset structure with extended metadata
    dataset = {
        'metadata': {
            'type': f'Composite DNA ({config["alphabet_mode"]})',
            'error_profile': f'{error_specs["name"]} ({error_specs["platform"]})',
            'error_model': config['error_model'],
            'platform': error_specs['platform'],
            'synthesis': error_specs['synthesis'],
            'reference': error_specs['reference'],
            'full_length': error_specs['full_length'],
            'index_length': error_specs['index_length'],
            'error_summary': errors.get_error_summary(),
            'num_samples': num_samples,
            'seq_length': seq_length,
            'coverage_depth': coverage,
            'vocab_size': config['vocab_size'],
            'alphabet_mode': config['alphabet_mode'],
            'symbols': list(symbol_to_idx.keys()),
            'symbol_to_idx': symbol_to_idx,
            'ideal_vectors': ideal_vectors,
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'seed': config['seed']
        },
        'data': []
    }
    
    print(f"\n{'='*60}")
    print(f"🔄 GENERATING DATASET")
    print(f"{'='*60}")
    print(f"   Error Model: {config['error_model']} ({error_specs['platform']})")
    print(f"   Oligo: {error_specs['full_length']}nt − {error_specs['index_length']}nt index = {seq_length}nt payload (standard)")
    print(f"   Samples: {num_samples:,}")
    print(f"   Sequence Length: {seq_length}")
    print(f"   Coverage Depth: {coverage}")
    print(f"   Alphabet: {config['alphabet_mode']} ({config['vocab_size']} classes)")
    print(f"{'='*60}")
    
    start_time = time.time()
    
    for i in range(num_samples):
        # A. Generate Ground Truth (Label) - composite sequence
        clean_composite_seq = generate_composite_sequence(seq_length, composite_map)
        
        # B. Generate Cluster (Input) - multiple noisy reads
        cluster_reads = []
        for _ in range(coverage):
            realized_dna = realize_sequence(clean_composite_seq, composite_map)
            noisy_read = apply_ids_noise(realized_dna, errors)
            cluster_reads.append(noisy_read)
            
        # C. Store sample
        sample = {
            'id': i,
            'label': clean_composite_seq,
            'cluster': cluster_reads
        }
        dataset['data'].append(sample)
        
        # Progress logging
        if (i + 1) % 10000 == 0:
            elapsed = time.time() - start_time
            samples_per_sec = (i + 1) / elapsed
            eta = (num_samples - i - 1) / samples_per_sec
            print(f"   Processed {i+1:,}/{num_samples:,} | "
                  f"Speed: {samples_per_sec:.1f} samples/s | "
                  f"ETA: {eta:.1f}s")

    # Save dataset
    with open(filename, 'wb') as f:
        pickle.dump(dataset, f)
    
    total_time = time.time() - start_time
    
    print(f"\n{'='*60}")
    print(f"✅ DATASET GENERATION COMPLETE")
    print(f"{'='*60}")
    print(f"   Error Model: {config['error_model']} ({error_specs['platform']})")
    print(f"   Output File: {filename}")
    print(f"   Total Time: {total_time:.1f}s ({total_time/60:.1f} min)")
    print(f"   File Size: {os.path.getsize(filename) / (1024*1024):.1f} MB")
    
    # Print example
    print(f"\n🔍 Sample 0:")
    print(f"   Label (first 10): {dataset['data'][0]['label'][:10]}")
    print(f"   Read 1 (first 20): {dataset['data'][0]['cluster'][0][:20]}...")
    
    return dataset


In [8]:
# In[8]:

# =============================================================================
# CELL 8: ANALYZE DATASET STATISTICS
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: dataset_generator_2mix_3mix-Erlich.py → Cell 8
# Copy: analyze_dataset() function
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

def analyze_dataset(dataset, symbol_to_idx):
    """Analyze and print dataset statistics."""
    
    print(f"\n{'='*60}")
    print(f"📊 DATASET STATISTICS")
    print(f"{'='*60}")
    
    all_symbols = []
    for sample in dataset['data']:
        all_symbols.extend(sample['label'])
    
    counter = Counter(all_symbols)
    total_symbols = len(all_symbols)
    
    print(f"\n🧬 Symbol Distribution in Labels:")
    print(f"   {'Symbol':<8} {'Count':>10} {'Percentage':>12} {'Expected':>10}")
    print(f"   {'-'*42}")
    
    num_classes = len(symbol_to_idx)
    expected_pct = 100.0 / num_classes
    
    for sym in sorted(counter.keys(), key=lambda x: symbol_to_idx[x]):
        count = counter[sym]
        pct = 100 * count / total_symbols
        print(f"   {sym:<8} {count:>10,} {pct:>11.2f}% {expected_pct:>9.2f}%")
    
    print(f"   {'-'*42}")
    print(f"   {'Total':<8} {total_symbols:>10,}")
    
    all_read_lengths = []
    for sample in dataset['data']:
        for read in sample['cluster']:
            all_read_lengths.append(len(read))
    
    print(f"\n📏 Read Length Statistics:")
    print(f"   Original Length: {dataset['metadata']['seq_length']}")
    print(f"   Mean Read Length: {np.mean(all_read_lengths):.2f}")
    print(f"   Std Read Length: {np.std(all_read_lengths):.2f}")
    print(f"   Min Read Length: {np.min(all_read_lengths)}")
    print(f"   Max Read Length: {np.max(all_read_lengths)}")
    
    pure_count = sum(counter.get(s, 0) for s in ['A', 'C', 'G', 'T'])
    two_mix_count = sum(counter.get(f'M{i}', 0) for i in range(1, 7))
    three_mix_count = sum(counter.get(f'T{i}', 0) for i in range(1, 5))
    four_mix_count = counter.get('Q1', 0)
    
    print(f"\n📈 Symbol Type Distribution:")
    print(f"   Pure Bases (A,C,G,T): {pure_count:,} ({100*pure_count/total_symbols:.1f}%)")
    print(f"   Two-Mix (M1-M6):      {two_mix_count:,} ({100*two_mix_count/total_symbols:.1f}%)")
    if three_mix_count > 0:
        print(f"   Three-Mix (T1-T4):    {three_mix_count:,} ({100*three_mix_count/total_symbols:.1f}%)")
    if four_mix_count > 0:
        print(f"   Four-Mix (Q1):        {four_mix_count:,} ({100*four_mix_count/total_symbols:.1f}%)")



In [9]:
# In[9]:

# =============================================================================
# CELL 9: ERROR PROFILE COMPARISON UTILITY
# =============================================================================

def print_all_profiles_comparison():
    """Print a comparison table of all available error profiles with oligo design details."""
    
    # All profiles with standard oligo design info
    all_specs = {
        "EZ17":  {"platform": "Illumina MiSeq + Twist",         "full": 152, "idx": 16, "n": 136},
        "G15":   {"platform": "Illumina MiSeq + CustomArray",    "full": 117, "idx": 13, "n": 104},
        "O17":   {"platform": "Illumina NextSeq + Twist",        "full": 110, "idx": 33, "n": 77},
        "R21":   {"platform": "Nanopore MinION + Twist",         "full": 152, "idx": 16, "n": 136},
        "B22":   {"platform": "Nanopore MinION Short + Twist",   "full": 152, "idx": 16, "n": 136},
        "BOS22": {"platform": "Illumina MiSeq 2022 + Twist",     "full": 152, "idx": 16, "n": 136},
        "NP22":  {"platform": "Nanopore Pilot 2022 + Twist",     "full": 152, "idx": 16, "n": 136},
        "NPF22": {"platform": "Nanopore Full 2022 + Twist",      "full": 152, "idx": 16, "n": 136},
    }
    
    print(f"\n{'='*115}")
    print(f"📊 CROSS-PLATFORM ERROR PROFILE COMPARISON (Standard Sequence Lengths)")
    print(f"{'='*115}")
    
    print(f"\n{'Profile':<8} {'Platform':<32} {'Oligo':>5} {'Idx':>4} {'n':>4} "
          f"{'Sub(%)':>8} {'Ins(%)':>8} {'Del(%)':>8} {'LDel(%)':>8} {'Total(%)':>9}")
    print(f"{'-'*115}")
    
    for name in ["EZ17", "G15", "O17", "R21", "B22", "BOS22", "NP22", "NPF22"]:
        err = ErrorRates()
        err.set_values_by_model(name)
        g = err.general_errors
        total = sum(g.values())
        sp = all_specs[name]
        
        print(f"{name:<8} {sp['platform']:<32} {sp['full']:>5} {sp['idx']:>4} {sp['n']:>4} "
              f"{g['s']*100:>8.4f} {g['i']*100:>8.4f} "
              f"{g['d']*100:>8.4f} {g['ld']*100:>8.4f} "
              f"{total*100:>9.4f}")
    
    print(f"\n   Illumina profiles: EZ17 (n=136), G15 (n=104), O17 (n=77), BOS22 (n=136)")
    print(f"   Nanopore profiles: R21 (n=136), B22 (n=136), NP22 (n=136), NPF22 (n=136)")
    print(f"   Note: R21/B22/BOS22/NP22/NPF22 share n=136 because they use the same")
    print(f"         Erlich/Twist 152nt oligo pool, re-sequenced with different technologies.")
    print(f"   Different sequence lengths across full set: n ∈ {{77, 104, 136}}")
    print(f"{'='*115}")

# Print comparison
print_all_profiles_comparison()



📊 CROSS-PLATFORM ERROR PROFILE COMPARISON (Standard Sequence Lengths)

Profile  Platform                         Oligo  Idx    n   Sub(%)   Ins(%)   Del(%)  LDel(%)  Total(%)
-------------------------------------------------------------------------------------------------------------------
   >> Loading Erlich (EZ17) Error Profile...
EZ17     Illumina MiSeq + Twist             152   16  136   0.1320   0.0581   0.0958   0.0233    0.3092
   >> Loading Grass (G15) Error Profile...
G15      Illumina MiSeq + CustomArray       117   13  104   0.5840   0.0857   0.5370   0.0348    1.2415
   >> Loading Organick (O17) Error Profile...
O17      Illumina NextSeq + Twist           110   33   77   0.2520   0.0414   0.0694   0.0211    0.3839
   >> Loading R21 (Nanopore MinION + Twist) Error Profile...
R21      Nanopore MinION + Twist            152   16  136   1.0800   1.6500   1.1800   0.3360    4.2460
   >> Loading B22 (Nanopore MinION Short + Twist) Error Profile...
B22      Nanopore MinION Short

In [10]:
# # In[10]:

# # =============================================================================
# # CELL 10: MAIN EXECUTION
# # =============================================================================

# if __name__ == "__main__":
    
#     print("\n" + "="*70)
#     print(f"🧬 COMPOSITE DNA DATASET GENERATOR - CROSS-PLATFORM")
#     print(f"   Error Model: {CONFIG['error_model']} ({CONFIG['error_specs']['name']})")
#     print(f"   Platform: {CONFIG['error_specs']['platform']}")
#     print(f"   Oligo: {CONFIG['error_specs']['full_length']}nt − {CONFIG['error_specs']['index_length']}nt = "
#           f"{CONFIG['error_specs']['seq_length']}nt (standard)")
#     print(f"   Alphabet: {CONFIG['alphabet_mode']}")
#     print("="*70)
    
#     # Check if dataset already exists
#     if os.path.exists(CONFIG['dataset_path']):
#         print(f"\n⚠️  Dataset already exists: {CONFIG['dataset_path']}")
#         response = input("   Overwrite? (y/n): ").strip().lower()
#         if response != 'y':
#             print("   Aborted.")
#             exit()
    
#     # Build mappings
#     COMPOSITE_MAP = build_composite_map(CONFIG["alphabet_mode"])
#     SYMBOL_TO_IDX = build_symbol_to_idx(CONFIG["alphabet_mode"])
#     IDEAL_VECTORS = build_ideal_vectors(CONFIG["alphabet_mode"])
#     ALL_SYMBOLS = list(COMPOSITE_MAP.keys())
    
#     # Generate dataset
#     dataset = generate_dataset(CONFIG, COMPOSITE_MAP, SYMBOL_TO_IDX, IDEAL_VECTORS)
    
#     # Analyze dataset
#     analyze_dataset(dataset, SYMBOL_TO_IDX)
    
#     print(f"\n{'='*70}")
#     print(f"🎉 Dataset generation completed successfully!")
#     print(f"   Error Model: {CONFIG['error_model']} ({CONFIG['error_specs']['platform']})")
#     print(f"   File: {CONFIG['dataset_path']}")
#     print(f"{'='*70}")

In [11]:
# In[11]:

# =============================================================================
# CELL 11: BATCH GENERATION HELPER (Optional)
# =============================================================================
# to generate datasets for ALL new profiles in one go.
# This will generate 5 profiles × N alphabet modes.


BATCH_PROFILES = ["R21", "B22", "BOS22", "NP22", "NPF22"]
BATCH_ALPHABETS = ["2mix_3mix", "2mix_3mix_4mix", "2mix_only"]  # or add "2mix_only"

for profile in BATCH_PROFILES:
    for alpha in BATCH_ALPHABETS:
        print(f"\n{'#'*70}")
        print(f"# GENERATING: {profile} × {alpha}")
        print(f"{'#'*70}")
        
        # Update config
        batch_config = CONFIG.copy()
        batch_config['error_model'] = profile
        batch_config['error_specs'] = ERROR_MODEL_SPECS[profile]
        batch_config['alphabet_mode'] = alpha
        batch_config['vocab_size'] = VOCAB_SIZES[alpha]
        batch_config['seq_length'] = ERROR_MODEL_SPECS[profile]['seq_length']
        
        dataset_name = f"dna_{profile}_{alpha}"
        batch_config['dataset_path'] = (
            f"{batch_config['dataset_dir']}/{dataset_name}_"
            f"{batch_config['num_samples']}_{batch_config['max_coverage']}.pkl"
        )
        
        # Build alphabet
        cmap = build_composite_map(alpha)
        sidx = build_symbol_to_idx(alpha)
        ivec = build_ideal_vectors(alpha)
        
        # Generate
        set_seed(batch_config['seed'])
        ds = generate_dataset(batch_config, cmap, sidx, ivec)
        analyze_dataset(ds, sidx)
        
        print(f"✅ Done: {profile} × {alpha}")



######################################################################
# GENERATING: R21 × 2mix_3mix
######################################################################
   >> Loading R21 (Nanopore MinION + Twist) Error Profile...

   --- Error Configuration ---
   General: sub=0.01080, ins=0.01650, del=0.01180, ldel=0.00336
   Total error rate: 0.04246 (4.246%)
   Base          Sub        Ins        Del       LDel
   ----------------------------------------------
   A         0.01039    0.01610    0.01192    0.00314
   C         0.01042    0.01639    0.01260    0.00334
   G         0.01094    0.01604    0.01267    0.00337
   T         0.01131    0.01738    0.01327    0.00357
   ----------------------------------------------

🔄 GENERATING DATASET
   Error Model: R21 (Oxford Nanopore MinION)
   Oligo: 152nt − 16nt index = 136nt payload (standard)
   Samples: 100,000
   Sequence Length: 136
   Coverage Depth: 25
   Alphabet: 2mix_3mix (14 classes)
   Processed 10,000/100,000 | Speed: 

   Processed 10,000/100,000 | Speed: 298.1 samples/s | ETA: 302.0s
   Processed 20,000/100,000 | Speed: 298.7 samples/s | ETA: 267.9s
   Processed 30,000/100,000 | Speed: 299.4 samples/s | ETA: 233.8s
   Processed 40,000/100,000 | Speed: 299.7 samples/s | ETA: 200.2s
   Processed 50,000/100,000 | Speed: 298.8 samples/s | ETA: 167.4s
   Processed 60,000/100,000 | Speed: 297.4 samples/s | ETA: 134.5s
   Processed 70,000/100,000 | Speed: 297.2 samples/s | ETA: 101.0s
   Processed 80,000/100,000 | Speed: 297.3 samples/s | ETA: 67.3s
   Processed 90,000/100,000 | Speed: 297.2 samples/s | ETA: 33.7s
   Processed 100,000/100,000 | Speed: 296.7 samples/s | ETA: 0.0s

✅ DATASET GENERATION COMPLETE
   Error Model: R21 (Oxford Nanopore MinION)
   Output File: ./dataset_cross_platform/dna_R21_2mix_only_100000_25.pkl
   Total Time: 338.8s (5.6 min)
   File Size: 360.6 MB

🔍 Sample 0:
   Label (first 10): ['C', 'A', 'M1', 'T', 'T', 'G', 'C', 'M5', 'C', 'M6']
   Read 1 (first 20): CATTTGCAGCAACTACAAA


🧬 Symbol Distribution in Labels:
   Symbol        Count   Percentage   Expected
   ------------------------------------------
   A           906,078        6.66%      6.67%
   C           906,995        6.67%      6.67%
   G           905,892        6.66%      6.67%
   T           907,623        6.67%      6.67%
   M1          905,883        6.66%      6.67%
   M2          908,638        6.68%      6.67%
   M3          907,052        6.67%      6.67%
   M4          906,155        6.66%      6.67%
   M5          907,633        6.67%      6.67%
   M6          906,718        6.67%      6.67%
   T1          907,525        6.67%      6.67%
   T2          905,593        6.66%      6.67%
   T3          906,532        6.67%      6.67%
   T4          906,378        6.66%      6.67%
   Q1          905,305        6.66%      6.67%
   ------------------------------------------
   Total    13,600,000

📏 Read Length Statistics:
   Original Length: 136
   Mean Read Length: 136.38
   Std Read Length: 

   Processed 10,000/100,000 | Speed: 308.1 samples/s | ETA: 292.1s
   Processed 20,000/100,000 | Speed: 308.0 samples/s | ETA: 259.7s
   Processed 30,000/100,000 | Speed: 306.3 samples/s | ETA: 228.5s
   Processed 40,000/100,000 | Speed: 307.0 samples/s | ETA: 195.4s
   Processed 50,000/100,000 | Speed: 307.5 samples/s | ETA: 162.6s
   Processed 60,000/100,000 | Speed: 307.8 samples/s | ETA: 130.0s
   Processed 70,000/100,000 | Speed: 307.3 samples/s | ETA: 97.6s
   Processed 80,000/100,000 | Speed: 307.6 samples/s | ETA: 65.0s
   Processed 90,000/100,000 | Speed: 307.8 samples/s | ETA: 32.5s
   Processed 100,000/100,000 | Speed: 307.7 samples/s | ETA: 0.0s

✅ DATASET GENERATION COMPLETE
   Error Model: BOS22 (Illumina MiSeq 2022)
   Output File: ./dataset_cross_platform/dna_BOS22_2mix_3mix_4mix_100000_25.pkl
   Total Time: 326.7s (5.4 min)
   File Size: 359.5 MB

🔍 Sample 0:
   Label (first 10): ['T1', 'C', 'A', 'T2', 'M1', 'T', 'T', 'G', 'T2', 'C']
   Read 1 (first 20): ACATTTTGCCATG


🧬 Symbol Distribution in Labels:
   Symbol        Count   Percentage   Expected
   ------------------------------------------
   A           971,076        7.14%      7.14%
   C           972,849        7.15%      7.14%
   G           971,301        7.14%      7.14%
   T           973,175        7.16%      7.14%
   M1          970,574        7.14%      7.14%
   M2          970,945        7.14%      7.14%
   M3          971,521        7.14%      7.14%
   M4          970,613        7.14%      7.14%
   M5          972,011        7.15%      7.14%
   M6          971,683        7.14%      7.14%
   T1          971,120        7.14%      7.14%
   T2          970,673        7.14%      7.14%
   T3          971,975        7.15%      7.14%
   T4          970,484        7.14%      7.14%
   ------------------------------------------
   Total    13,600,000

📏 Read Length Statistics:
   Original Length: 136
   Mean Read Length: 136.20
   Std Read Length: 1.69
   Min Read Length: 126
   Max Read Length

   Processed 10,000/100,000 | Speed: 312.6 samples/s | ETA: 287.9s
   Processed 20,000/100,000 | Speed: 310.9 samples/s | ETA: 257.3s
   Processed 30,000/100,000 | Speed: 311.3 samples/s | ETA: 224.9s
   Processed 40,000/100,000 | Speed: 311.2 samples/s | ETA: 192.8s
   Processed 50,000/100,000 | Speed: 311.5 samples/s | ETA: 160.5s
   Processed 60,000/100,000 | Speed: 310.6 samples/s | ETA: 128.8s
   Processed 70,000/100,000 | Speed: 309.7 samples/s | ETA: 96.9s
   Processed 80,000/100,000 | Speed: 309.1 samples/s | ETA: 64.7s
   Processed 90,000/100,000 | Speed: 308.8 samples/s | ETA: 32.4s
   Processed 100,000/100,000 | Speed: 307.8 samples/s | ETA: 0.0s

✅ DATASET GENERATION COMPLETE
   Error Model: NPF22 (Nanopore Full Pool Nov-2022)
   Output File: ./dataset_cross_platform/dna_NPF22_2mix_3mix_100000_25.pkl
   Total Time: 326.5s (5.4 min)
   File Size: 360.2 MB

🔍 Sample 0:
   Label (first 10): ['T1', 'C', 'A', 'T2', 'M1', 'T', 'T', 'G', 'T2', 'C']
   Read 1 (first 20): GCATTTTGAC


🧬 Symbol Distribution in Labels:
   Symbol        Count   Percentage   Expected
   ------------------------------------------
   A         1,359,651       10.00%     10.00%
   C         1,359,705       10.00%     10.00%
   G         1,358,241        9.99%     10.00%
   T         1,360,294       10.00%     10.00%
   M1        1,359,389       10.00%     10.00%
   M2        1,360,149       10.00%     10.00%
   M3        1,360,449       10.00%     10.00%
   M4        1,361,109       10.01%     10.00%
   M5        1,360,004       10.00%     10.00%
   M6        1,361,009       10.01%     10.00%
   ------------------------------------------
   Total    13,600,000

📏 Read Length Statistics:
   Original Length: 136
   Mean Read Length: 136.31
   Std Read Length: 1.74
   Min Read Length: 127
   Max Read Length: 146

📈 Symbol Type Distribution:
   Pure Bases (A,C,G,T): 5,437,891 (40.0%)
   Two-Mix (M1-M6):      8,162,109 (60.0%)
✅ Done: NPF22 × 2mix_only
